In [ ]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModel

In [ ]:
# loading the dataset
issues_dataset = load_dataset("lewtun/github-issues", split="train")
issues_dataset

In [ ]:
# filtering out pill requests
issues_dataset = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0)
)
issues_dataset

In [ ]:
# filtering out unnecessary columns
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

In [ ]:
# converting to the dataframe
issues_dataset.set_format("pandas")
df = issues_dataset[:]

In [ ]:
df["comments"][0].tolist()

In [ ]:
df.head(4)

In [ ]:
# "comments" column contains the list of comments. using .explode to make a row for each comment (i.e. exploding the list of comments)
comments_df = df.explode("comments", ignore_index=True)
comments_df.head(4)

In [ ]:
# converting back to the dataset
comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

In [ ]:
# adding comments lenght as a column
comments_dataset = comments_dataset.map(
    lambda x: {"comment_length": len(x["comments"].split())}
)

In [ ]:
# filtering out short comments (e.g. "Thanks!")
comments_dataset = comments_dataset.filter(lambda x: x["comment_length"] > 15)
comments_dataset

In [ ]:
def concatenate_text(examples):
    """
    A function to concatenate
    title, body and comments
    """
    return {
        "text": examples["title"]
        + " \n"
        + examples["body"]
        + " \n"
        + examples["comments"]
    }

In [ ]:
comments_dataset = comments_dataset.map(concatenate_text)

In [ ]:
# loading models that will be used to create embeddings
model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

In [ ]:
# switching to gpu
device = torch.device("cuda")
model.to(device)

In [ ]:
def cls_pooling(model_output):
    """
    Performs CLS polling on embeddings
    """
    return model_output.last_hidden_state[:, 0]

In [ ]:
def get_embeddings(text_list):
    """
    Gets the embeddings from a text
    """
    encoded_input = tokenizer(
        text_list, padding=True, truncation=True, return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

In [ ]:
# creating embeddings for the dataset
embeddings = get_embeddings(comments_dataset["text"][0])
embeddings.shape

In [ ]:
# adding embeddings to the dataset
embeddings_dataset = comments_dataset.map(
    lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

In [ ]:
embeddings_dataset

In [ ]:
# creating faiss index for each embedding
embeddings_dataset.add_faiss_index(column="embeddings")

In [ ]:
# testing with a question
question = "How can I load a dataset offline?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

In [ ]:
# finding top 5 best matches for the answer
scores, samples = embeddings_dataset.get_nearest_examples(
    "embeddings", question_embedding, k=5
)

In [ ]:
# making a dataframe to check the answers
samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)

In [ ]:
# displaying the answers
for _, row in samples_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.scores}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()